In [2]:
import random
import math
import pygame
import sys

pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.7)


In [ ]:
class PerudoGame:

    def __init__(self, players = 2, dice_per_player = 5, debug_mode = False, max_dice = 5):
        
        self.players = []
        self.dice_per_player = dice_per_player
        
        for i in range(players):
            self.players.append(Player(f"Player {i}",dice_per_player))

        self.game_over = False
        self.max_dice = max_dice
        self.debug_mode = debug_mode
        self.previous_bids = []
        self.current_player = None
        self.current_bid = None
        self.challenger = None
        self.next_start_player = None
        self.is_palifico = False
        
        

#   UI and debug functions
    def get_dice_counts(self):
        dice_count = {}
        for player in self.players:
            dice_count[player.name] = len(player.dice)
        return dice_count
    
    def reset(self):
        
        for player in self.players:
            player.reset()
        self.previous_bids = []
    
        
        self.current_player = self.players[0]
        self.next_start_player = None
        self.current_bid = None
        self.challenger = None

#   game loop functions
    def next_round(self):
        self.remove_players_with_no_dice()
        if self.check_is_winner():
            return
        self.current_player = self.next_start_player
        self.previous_bids = []
        self.current_bid = None
        self.players_roll()
        self.check_palifico()
        
    def players_roll(self):
        for player in self.players:
            player.roll()
    def check_palifico(self):
        if len(self.current_player.dice) == 1 and self.current_player.palifico == False:
            self.current_player.palifico = True
            self.is_palifico = True
        else:
            self.is_palifico = False
    def perform_action(self, action):
            if action.action_type == "bid":
                if self.is_bid_legal(action):
                    self.apply_bid(action)
                    self.next_player()
                    return True
                else:
                    print("Invalid bid. Please try again.")
                    return False
            elif action.action_type == "call":
                self.resolve_call()
                self.next_round()
            elif action.action_type == "perudo":
                print(f"Perudo Called on {self.current_bid.quantity}, {self.current_bid.face}'s")
                self.resolve_perudo()
                self.next_round()
            else:
                raise ValueError(f"Unknown action type: {action.action_type}")
            return True
    def next_player(self):
        current_index = self.players.index(self.current_player)
        self.current_player = self.players[(current_index + 1) % len(self.players)]
        #self.current_player.make_action
    def remove_players_with_no_dice(self):
        for player in self.players.copy():
            if len(player.dice) == 0:
                self.players.remove(player)
    def check_is_winner(self):
        if len(self.players) == 1:
            print(f"{self.players[0].name} has won")
            self.game_over = True
            return True
        return False

#   bid creation and testing functions   
    def is_bid_legal(self, action):
        
        if 1> action.face > 6 or action.quantity < 1:
            return False
        elif self.current_bid is None:
            return True
        elif self.is_palifico:
            return self.is_bid_legal_palifico(action)
        elif self.current_bid.face == 1:
            return self.is_leaving_ones_bid_legal(action)
        elif action.face == 1:
            return self.is_switch_to_ones_legal(action)
        return self.is_normal_bid_legal(action)
    def is_bid_legal_palifico(self, action):
        return action.face == self.current_bid.face and action.quantity > self.current_bid.quantity    
    def is_leaving_ones_bid_legal(self, action):
        if action.face == 1:
            return action.quantity > self.current_bid.quantity
        return action.quantity >= self.current_bid.quantity * 2 and self.check_previous_bids(action)
    def is_normal_bid_legal(self, action):
        return (
            action.quantity > self.current_bid.quantity 
            or action.face > self.current_bid.face
            )
    def is_switch_to_ones_legal(self, action):
        return (
            self.check_previous_bids(action) 
            and (action.quantity == (self.current_bid.quantity+1)//2 
            and action.face == 1)
              )
    def get_previous_non_one_bid(self):
        for bid in reversed(self.previous_bids):
            if bid.face != 1:
                return bid   
    def check_previous_bids(self, action):
        if action.face == 1:
            for previous_action in self.previous_bids:
                if previous_action.quantity == action.quantity and previous_action.face == action.face:
                    return False
        if action.face != 1:
            previous_bid = self.get_previous_non_one_bid()
            if action.quantity>previous_bid.quantity or action.face > previous_bid.face:
                return True
        return True
    def apply_bid(self, action):
        self.current_bid = action
        self.previous_bids.append(action)
        pass

#   calling bluff functions
    def count_face(self, face):
        count = 0
        for player in self.players:
            count += player.dice.count(face)
            # 1s are jokers unless el palifico rules apply
            if face != 1 and self.is_palifico == False:
                count += player.dice.count(1)
        return count
    def is_call_false(self):
        return self.count_face(self.current_bid.face)>=self.current_bid.quantity
    def resolve_call(self):
        challenger = self.current_player
        if self.is_call_false():
            self.next_start_player = challenger
            challenger.remove_die()
        else:
            self.next_start_player = self.current_bid.player
            self.current_bid.player.remove_die()

#   perudo function
    def resolve_perudo(self):
        self.next_start_player = self.current_player
        if self.count_face(self.current_bid.face) == self.current_bid.quantity:
            if len(self.current_player.dice) < self.max_dice:
                self.current_player.add_die()
            print(f"{self.current_player.name}'s perudo was succesful")
        else:
            self.current_player.remove_die()
            print(f"{self.current_player.name}'s perudo was unsuccesful")  

In [4]:
class Player:
    
    def __init__(self, name: str, initial_dice_quantity):
        
        self.name = name
        self.initial_dice_quantity = initial_dice_quantity
        self.dice = []
        self.palifico = False
        self.reset()
        
    def reset(self):
            self.dice = []
            for i in range(self.initial_dice_quantity):
                        self.dice.append(random.randint(1,6))


    # control functions
    def remove_die(self):
        self.dice.pop()
    def add_die(self):
        self.dice.append(1)
    def roll(self):
        for i in range(len(self.dice)):
            self.dice[i] = random.randint(1,6)
    
    
    # turn actions
    def make_bid(self, quantity, face):
        return Action("bid", self, quantity, face)
    def call_bluff(self):
        return Action("call", self)
    def call_perudo(self):
        return Action("perudo", self)

In [5]:
class Action:
    def __init__(self, action_type, player, quantity = None, face = None):
        self.action_type = action_type
        self.player = player
        self.quantity = quantity
        self.face = face

In [ ]:
class PlayerUI:
    
    def __init__(self, player = Player, position = (0,0), name_offset = -20, dice_offset = 20):
        self.position = position
        self.player = player
        self.default_name_offset = name_offset
        self.default_dice_offset = dice_offset
    
    def draw_name(self, surface):
        name = self.player.name
        text = font.render(name.upper(), True, white)

        text_rect = text.get_rect(
            midbottom = (self.position[0],self.position[1]+self.default_name_offset)
        )
        surface.blit(text, text_rect)

    def draw_icon(self, surface):
        x, y = self.position
        player_icon = pygame.image.load("player_icon.png").convert_alpha()
        player_icon = pygame.transform.scale(player_icon, (32,32))
        player_icon_rect = player_icon.get_rect(
            center = (x,y)
        )
        
        width, height = player_icon.get_size()

        border = pygame.Rect(0,0,width,height)
        border.center = (x,y)
        pygame.draw.rect(surface, white, border, border_radius = 2)
        pygame.draw.rect(surface, black, border, width = 2, border_radius = 2)
        surface.blit(player_icon, player_icon_rect)

    def draw_dice(self, surface, dice_size = 20, dice_spacing = 3):
        dice = self.player.dice
        surface_width = (dice_size + dice_spacing) * len(dice) + dice_spacing
        surface_height = (dice_size)
        dice_surface = pygame.Surface((surface_width,surface_height), pygame.SRCALPHA)
        count = 0
        for die in dice:
            die_x = ((count+1)*dice_spacing) + count*dice_size
            die_y = 0
            draw_die(dice_surface, (die_x, die_y),dice_size, die)
            count += 1
        dice_surface_rect = dice_surface.get_rect(
            midtop = (self.position[0], self.position[1] + self.default_dice_offset)
        )
        surface.blit(dice_surface, dice_surface_rect)

In [ ]:
def pos_around_table(degrees):
    x = table_centre_x + table_radius_x*math.cos(math.radians(90 + degrees))
    y = table_centre_y + table_radius_y*math.sin(math.radians(90 + degrees))
    return x, y
def player_pos(number_of_players, player_index):
    return pos_around_table((360/number_of_players)*player_index)
dice_dot_positions = {
    1: [(0.5,0.5)],
    2: [(0.25,0.25),(0.75,0.75)],
    3: [(0.25,0.25),(0.5,0.5),(0.75,0.75)],
    4: [(0.25,0.25),(0.25,0.75),(0.75,0.25),(0.75,0.75)],
    5: [(0.25,0.25),(0.25,0.75),(0.5,0.5),(0.75,0.25),(0.75,0.75)],
    6: [(0.25,0.25),(0.25,0.75),(0.25,0.5),(0.75,0.5),(0.75,0.25),(0.75,0.75)]
}
def draw_die(surface, position=(0,0), size = 20, number= 1, show= True):
    die = pygame.Surface((size,size), pygame.SRCALPHA)
    
    if show:
        pygame.draw.rect(die, white, (0, 0, size, size), border_radius = int(size*0.1))
        pygame.draw.rect(die, black, (0, 0, size, size), width=int(size*0.1), border_radius = int(size*0.1))
        for i in range(number):
            rel_x, rel_y = dice_dot_positions[number][i]
            pygame.draw.circle(die, black, (int(rel_x*size),
                                            int(rel_y*size)),size*0.1)
    else:
        pygame.draw.rect(die, black, (0, 0, size, size), border_radius = int(size*0.1))
    surface.blit(die,position)
def draw_players(screen):
    count = 0
    number_of_players = len(game.players)
    for player in game.players:
        player_UI = PlayerUI(player, player_pos(number_of_players,count))
        player_UI.draw_dice(screen)
        player_UI.draw_name(screen)
        player_UI.draw_icon(screen)
        count += 1

In [ ]:
game = PerudoGame(4,5)
pygame.init()

screen_size = width, height = 720,480
black = 0,0,0
green = 80, 130, 90
white = 255,255,255
background = 125,111,76
font_size = 16
font = pygame.font.SysFont("Arial", font_size)
screen = pygame.display.set_mode(screen_size)
clock = pygame.time.Clock()
running = True


# table variables
table_offset_x = 0
table_offset_y = screen_size[1]/7
table_width = 500
table_height = 200
table_centre_x = screen_size[0]/2 + table_offset_x
table_centre_y = screen_size[1]/2 + table_offset_y
table_radius_x = table_width/2
table_radius_y = table_height/2
table_x = table_centre_x - table_radius_x
table_y = table_centre_y - table_radius_y




try:
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        if not running:
            break
        screen.fill(background)
        pygame.draw.ellipse(screen, green, (table_x, table_y, table_width, table_height))
        pygame.draw.ellipse(screen, black, (table_x, table_y, table_width + 5, table_height + 5), width = 5)


        
        draw_players(screen)
        
        

        
        pygame.display.flip()
        clock.tick(60)
finally:
    pygame.quit()